# Example 2.2: 参数化工况模板 + 共享 base INP

主 INP 是一个带 `{{placeholder}}` 的工况模板, 它 `*INCLUDE` 了一个静态的 base INP(网格/几何):

```
planar_stress_scenario_template.inp   # 根模板, 含 {{youngs_modulus}} 和 {{load_magnitude}}
└── planar_stress_main.inp            # 静态, 125 KB, 全批次共用
```

**用法和 `01_Batch_wo_include.ipynb` 完全一样** —— 仍然是 `kind="inp_based"` + `generate_from_array`,
不需要为 `*INCLUDE` 做任何额外配置。

准备阶段会自动把 `*INCLUDE` 的相对路径改写成**绝对路径**: 生成的 INP 落在 job 目录里,
而原来的相对路径是相对于模板所在目录写的, 在 job 目录下解析不到。

base INP **不会**被复制进 job 目录, 也**不会**被内联进主 INP —— `*INCLUDE` 指令仍然是 `*INCLUDE`,
变的只有 `INPUT=` 后面那个路径。N 个 job 共指同一份文件, 最后一个 cell 可以验证这一点。

In [1]:
import os
import numpy as np

from ABQflow import BatchAbaqusProcessor, JobSpec, PreparationSpec, HookSpec
from ABQflow import generate_from_array, degenerate_from_array

%reload_ext autoreload
%autoreload 2

ABAQUS_CAE = 'C:/Applications/SIMULIA/Commands/2026/abaqus.bat'
CWD = os.getcwd()

In [2]:
param_names = ['youngs_modulus', 'load_magnitude']
param_values = np.array([
	[200000, 2000],
	[210000, 3000],
	[220000, 4000],
	[230000, 5000]
])

base_job_spec = JobSpec(
	job_name = "planar_stress_include",
	workflow = "modular",
	preparation = PreparationSpec(
		kind = "inp_based",
		# 根模板自己带 {{}}, 同时 *INCLUDE 了 planar_stress_main.inp
		source_path = "./examples/cae_file/planar_stress_scenario_template.inp",
		# include 树默认就会被解析; 只有在你想让 *INCLUDE 原样保留时才需要
		# options = {'resolve_includes': False}
	),
	pre_extraction = [
		HookSpec(
			script_path = "./examples/extraction_scripts/get_total_mass.py",
			tasks = [
				{"result_name": "total_mass",},
			]
		)
	],
	post_extraction = [
		HookSpec(
			script_path = "./examples/extraction_scripts/get_max_stress_mises.py",
			tasks = [
				{"result_name": "max_stress_mises",},
				{"result_name": "max_displacement",},
			]
		)
	]
)

spec_list = generate_from_array(
	samples_array = param_values,
	param_names = param_names,
	base_spec  = base_job_spec
)

import pprint

pprint.pprint(spec_list)

[JobSpec(job_name='planar_stress_include_0001',
         workflow='modular',
         preparation=PreparationSpec(kind='inp_based',
                                     source_path='./examples/cae_file/planar_stress_scenario_template.inp',
                                     params={'load_magnitude': 2000.0,
                                             'youngs_modulus': 200000.0},
                                     options={}),
         preflight=None,
         monolithic_script=None,
         monolithic_params={},
         pre_extraction=[HookSpec(script_path='./examples/extraction_scripts/get_total_mass.py',
                                  tasks=[{'result_name': 'total_mass'}])],
         post_extraction=[HookSpec(script_path='./examples/extraction_scripts/get_max_stress_mises.py',
                                   tasks=[{'result_name': 'max_stress_mises'},
                                          {'result_name': 'max_displacement'}])],
         subroutine=None,
         meta

In [3]:
processor = BatchAbaqusProcessor(
	batch_data = spec_list,
	base_output_dir = os.path.join(CWD, "examples/02_BatchParameterizedJob/output_include"),
	cpus_per_job = 12,
	duplicate_mode = "overwrite",
	abaqus_exe = ABAQUS_CAE,
)

In [4]:
outcomes = processor.run_batch(
	num_parallel_jobs = 2,
)

Output()

In [5]:
outcomes

[JobOutcome(job_name='planar_stress_include_0002', status='COMPLETED', results={'total_mass': 0.00032066262255247625, 'max_stress_mises': 6787.890625, 'max_displacement': 5.984342098236084}, error=None, diagnostics=None, output_dir='c:\\SJTU\\Projects_Code\\24_Abaqus_Pack\\examples/02_BatchParameterizedJob/output_include\\planar_stress_include_0002', phases=[{'phase': 'preparation', 'status': 'PREPARATION_SUCCESS', 'started_at': 1788594043.1911817, 'ended_at': 1788594043.1941812, 'duration_s': 0.002999544143676758, 'error': None}, {'phase': 'pre_extraction', 'status': 'EXTRACTION_SUCCESS', 'started_at': 1788594043.1941812, 'ended_at': 1788594046.1155992, 'duration_s': 2.9214179515838623, 'error': None}, {'phase': 'simulation', 'status': 'SIMULATION_SUCCESS', 'started_at': 1788594046.1155992, 'ended_at': 1788594098.7765043, 'duration_s': 52.66090512275696, 'error': None}, {'phase': 'post_extraction', 'status': 'EXTRACTION_SUCCESS', 'started_at': 1788594098.7765043, 'ended_at': 178859410

In [6]:
Y = degenerate_from_array(
	outcomes = outcomes,
	output_names = ["total_mass", "max_stress_mises", "max_displacement"],
)
print(f"output: {Y}")

output: [[3.20662623e-04 4.52526025e+03 4.18903971e+00]
 [3.20662623e-04 6.78789062e+03 5.98434210e+00]
 [3.20662623e-04 9.05052051e+03 7.61643553e+00]
 [3.20662623e-04 1.13131514e+04 9.10660839e+00]]


In [7]:
# 验证: 准备阶段到底往 job 目录里放了什么
job_dir = outcomes[0].output_dir
job_inp = os.path.join(job_dir, outcomes[0].job_name + ".inp")

print("job 目录里的 INP 文件:")
for name in sorted(os.listdir(job_dir)):
	if name.endswith(".inp"):
		size = os.path.getsize(os.path.join(job_dir, name))
		print(f"  {name:<45s} {size:>9,d} bytes")

with open(job_inp, encoding="utf-8", errors="replace") as f:
	text = f.read()

print("\n生成的主 INP 里的 *INCLUDE 指令:")
for line in text.splitlines():
	if line.lstrip().lower().startswith("*include"):
		print("  " + line)

shared = "./examples/cae_file/planar_stress_main.inp"
print(f"\n共享的 base INP: {os.path.getsize(shared):,d} bytes")
print("它既没有被复制进 job 目录, 也没有被内联进主 INP —— 只是被绝对路径指向。")


job 目录里的 INP 文件:
  planar_stress_include_0002.inp                      883 bytes

生成的主 INP 里的 *INCLUDE 指令:
  *Include, input=c:\SJTU\Projects_Code\24_Abaqus_Pack\examples\cae_file\planar_stress_main.inp

共享的 base INP: 125,469 bytes
它既没有被复制进 job 目录, 也没有被内联进主 INP —— 只是被绝对路径指向。
